<a href="https://colab.research.google.com/github/2303A51529/ARVR-Project/blob/main/Air_Pollution_Prediction_Using_Evidential_Deep_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =====================================================================
# Air Pollution Prediction Using Evidential Deep Regression (EDR)
# Student Hall Ticket Number: 2303A51529
# Model Assigned: Evidential Deep Regression
# Target: PM2.5 | Features: RH, WD, SR, BP, AT
# =====================================================================

import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set seed for exact reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# 1. Load Data
excel_file = 'Data.xlsx'
print(f"Loading data from {excel_file}...")
df_raw_path = "/content/Data.xlsx"
df_raw = pd.read_excel(df_raw_path)
features = ['RH', 'WD', 'SR', 'BP', 'AT']
target = 'PM2.5'
df = df_raw[[target] + features].copy()

# 2. Train/Test Split (80% Train, 20% Test)
train_full, test = train_test_split(df, test_size=0.20, random_state=42)

# Impute missing Solar Radiation strictly with training median (prevent leakage)
sr_median = train_full['SR'].median()
train_full['SR'] = train_full['SR'].fillna(sr_median)
test['SR'] = test['SR'].fillna(sr_median)

# Split train_full into train (85%) and validation (15%)
train, val = train_test_split(train_full, test_size=0.15, random_state=42)

# 3. Standardization (Fit ONLY on training set)
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(train[features].values)
X_val = scaler_X.transform(val[features].values)
X_test = scaler_X.transform(test[features].values)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(train[[target]].values).flatten()
y_val = scaler_y.transform(val[[target]].values).flatten()
y_test = scaler_y.transform(test[[target]].values).flatten()
y_test_orig = test[target].values

# 4. Evidential Deep Regression Architecture
class EvidentialRegressionNet(nn.Module):
    def __init__(self, in_features=5, hidden_units=128, dropout_rate=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_units),
            nn.LayerNorm(hidden_units),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_units, hidden_units),
            nn.LayerNorm(hidden_units),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_units, 64),
            nn.LayerNorm(64),
            nn.SiLU()
        )
        self.out = nn.Linear(64, 4)

    def forward(self, x):
        h = self.net(x)
        raw = self.out(h)
        gamma = raw[:, 0]                         # Mean prediction (unrestricted)
        v = F.softplus(raw[:, 1]) + 1e-4          # Virtual count (v > 0)
        alpha = F.softplus(raw[:, 2]) + 1.0 + 1e-4# Shape parameter (alpha > 1)
        beta = F.softplus(raw[:, 3]) + 1e-4       # Scale parameter (beta > 0)
        return gamma, v, alpha, beta

# 5. Evidential Loss (Student-t NLL + Evidence Regularization)
def evidential_loss(gamma, v, alpha, beta, y, reg_coeff=0.01):
    diff = y - gamma
    error = torch.abs(diff)
    omega = 2.0 * beta * (1.0 + v)

    term1 = 0.5 * torch.log(math.pi / v)
    term2 = -alpha * torch.log(omega)
    term3 = (alpha + 0.5) * torch.log(v * (diff ** 2) + omega)
    term4 = torch.lgamma(alpha) - torch.lgamma(alpha + 0.5)

    nll = term1 + term2 + term3 + term4
    reg = error * (2.0 * v + alpha)
    return torch.mean(nll + reg_coeff * reg)

# 6. Training Pipeline
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)),
    batch_size=64, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)),
    batch_size=64, shuffle=False
)

model = EvidentialRegressionNet(in_features=5, hidden_units=128, dropout_rate=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=140)

best_loss = float('inf')
best_weights = None

print("Training Evidential Deep Regression model...")
for epoch in range(1, 141):
    model.train()
    for bx, by in train_loader:
        optimizer.zero_grad()
        g, v, a, b = model(bx)
        loss = evidential_loss(g, v, a, b, by, reg_coeff=0.01)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
    scheduler.step()

    # Validation
    model.eval()
    val_loss, count = 0.0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            g, v, a, b = model(bx)
            vl = evidential_loss(g, v, a, b, by, reg_coeff=0.01)
            val_loss += vl.item()
            count += 1
    val_loss /= count
    if val_loss < best_loss:
        best_loss = val_loss
        best_weights = model.state_dict().copy()

if best_weights is not None:
    model.load_state_dict(best_weights)

# 7. Evaluation on Untouched Test Set
model.eval()
with torch.no_grad():
    g_s, v_t, a_t, b_s = model(torch.tensor(X_test, dtype=torch.float32))
    gamma_np = g_s.numpy()
    v_np = v_t.numpy()
    alpha_np = a_t.numpy()
    beta_np = b_s.numpy()

# Inverse Scaling
sigma_y = scaler_y.scale_[0]
mu_y = scaler_y.mean_[0]

y_pred = gamma_np * sigma_y + mu_y
beta_orig = beta_np * (sigma_y ** 2)

# Uncertainty Decomposition
aleatoric_var = beta_orig / (alpha_np - 1.0)
epistemic_var = beta_orig / (v_np * (alpha_np - 1.0))
total_var = aleatoric_var + epistemic_var

aleatoric_std = np.sqrt(aleatoric_var)
epistemic_std = np.sqrt(epistemic_var)
total_std = np.sqrt(total_var)

# Metrics
mse = mean_squared_error(y_test_orig, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_orig, y_pred)
r2 = r2_score(y_test_orig, y_pred)
mape = np.mean(np.abs((y_test_orig - y_pred) / y_test_orig)) * 100
mdape = np.median(np.abs((y_test_orig - y_pred) / y_test_orig)) * 100
coverage = np.mean((y_test_orig >= y_pred - 1.96 * total_std) & (y_test_orig <= y_pred + 1.96 * total_std)) * 100

print("\n" + "="*50)
print("TEST EVALUATION RESULTS (Untouched 20% Holdout):")
print("="*50)
print(f"Mean Squared Error (MSE)       : {mse:.2f}")
print(f"Root Mean Squared Error (RMSE) : {rmse:.2f} ug/m3")
print(f"Mean Absolute Error (MAE)      : {mae:.2f} ug/m3")
print(f"R-squared Score (R2)           : {r2:.4f}")
print(f"Mean Abs Percentage Error(MAPE): {mape:.2f}%")
print(f"Median Abs Percentage Err(MdAPE):{mdape:.2f}%")
print(f"95% Prediction Interval Cover. : {coverage:.2f}%")
print(f"Mean Aleatoric Std (sigma_ale) : {np.mean(aleatoric_std):.2f} ug/m3")
print(f"Mean Epistemic Std (sigma_epi) : {np.mean(epistemic_std):.2f} ug/m3")
print(f"Mean Total Predictive Std      : {np.mean(total_std):.2f} ug/m3")
print("="*50)

Loading data from Data.xlsx...
Training Evidential Deep Regression model...

TEST EVALUATION RESULTS (Untouched 20% Holdout):
Mean Squared Error (MSE)       : 2261.43
Root Mean Squared Error (RMSE) : 47.55 ug/m3
Mean Absolute Error (MAE)      : 32.17 ug/m3
R-squared Score (R2)           : 0.5480
Mean Abs Percentage Error(MAPE): 68.80%
Median Abs Percentage Err(MdAPE):17.16%
95% Prediction Interval Cover. : 91.53%
Mean Aleatoric Std (sigma_ale) : 16.66 ug/m3
Mean Epistemic Std (sigma_epi) : 33.99 ug/m3
Mean Total Predictive Std      : 38.07 ug/m3
